# SPARTAN vs IMPROVE — follow-up plots (mass space + per-site ratios + iron + ETBI)

Follow-ups from the 2026-06 meeting with Ann:

1. **Replot the network cross-plots in mass on filter (µg), not µg/m³.** Analysis techniques only see what is on the filter, so comparisons across networks must remove sampled volume. Volume-free quantities (ratios) are unchanged.
2. **fAbs / EC ratio by site** — find IMPROVE sites with an Addis-like effective MAC, instead of only network-wide boxplots.
3. **OC / EC ratio by site** — the relevant view for Satoshi's hypothesis: if a site's OC/EC is unusually low, FTIR may not be able to use its functional-group calibration well, and FTIR EC could be off for that reason rather than because of char.
4. **HIPS fAbs vs iron (no EC in the plot)** — confirm we still see no apparent iron effect in the new mass framing.
5. **Cross-plots for the new ETBI (Bishoftu, Ethiopia) site** — HIPS only so far (no ChemSpec/FTIR yet), so the useful views are fAbs distributions/time series against ETAD and the rest of the network.

Data sources:
- SPARTAN: `research/ftir_hips_chem/Filter Data/unified_filter_dataset.pkl` (4 sites with FTIR+HIPS+ChemSpec) and the HIPS batch file `SPARTAN_HIPS_Batch1-51.v2.csv` (all HIPS sites incl. ETBI/INJA/AEAZ) from the Google Drive pull.
- IMPROVE: FED Query Wizard chemistry export (`ahzs645_20260422_122542_N0100.xlsx`) from the same Drive pull — `ECf_Val`, `OCf_Val`, `fAbs_Val`, `FEf_Val`, `MF_Val`, `FlowRate_Val`, `SampDur_Val`.


## 0. Setup

In [ ]:
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "AGENTS.md").exists() and (cand / "research").exists():
            return cand
    raise RuntimeError("Could not locate repo root (AGENTS.md). CWD=%s" % Path.cwd())


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "research/ftir_hips_chem/scripts"))
try:
    from plotting import apply_default_style
    from plotting.utils import calculate_regression_stats
    apply_default_style()
except Exception as e:
    print("plotting package unavailable (%s); using fallbacks" % e)

    def calculate_regression_stats(x, y):
        x = np.asarray(x, float); y = np.asarray(y, float)
        m = np.isfinite(x) & np.isfinite(y)
        x, y = x[m], y[m]
        if len(x) < 3:
            return None
        slope, intercept = np.polyfit(x, y, 1)
        r = np.corrcoef(x, y)[0, 1]
        return {"n": len(x), "slope": slope, "intercept": intercept,
                "r_squared": r ** 2, "correlation": r}

# --- paths ---
SPARTAN_PKL = REPO_ROOT / "research/ftir_hips_chem/Filter Data/unified_filter_dataset.pkl"
DRIVE = REPO_ROOT / "research/improve_hips_offset/data_improve/Data"
IMPROVE_XLSX = DRIVE / "Improve/ahzs645_20260422_122542_N0100.xlsx"
HIPS_BATCH_CSV = DRIVE / "Spartan/SPARTAN_HIPS_Batch1-51.v2.csv"
SITE_LOOKUP_XLSX = DRIVE / "Spartan/SPARTAN_Site_quick_lookup.xlsx"
IMPROVE_CACHE = DRIVE / "Improve/improve_chemistry_clean_cache.pkl"  # gitignored with the rest of data_improve

OUT = REPO_ROOT / "research/spartan/follow_up_plots"
FIG_DIR = OUT / "figures"
TAB_DIR = OUT / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

NET_COLORS = {"SPARTAN": "#C0392B", "IMPROVE": "#2E86C1"}
SITE_COLORS = {"CHTS": "#8E44AD", "ETAD": "#E67E22", "INDH": "#16A085",
               "USPA": "#2C3E50", "ETBI": "#D35400"}
SITE_NAMES = {"CHTS": "Beijing", "ETAD": "Addis Ababa", "INDH": "Delhi",
              "USPA": "Pasadena/JPL", "ETBI": "Bishoftu"}
MIN_SITE_N = 50          # min samples for an IMPROVE site to enter the per-site ratio views
SCATTER_SAMPLE = 8000    # cap on IMPROVE points drawn in scatters (fits use all rows)
RNG = np.random.default_rng(0)

print("Repo root:", REPO_ROOT)
print("Figures ->", FIG_DIR)
print("Tables  ->", TAB_DIR)

## 1. Load IMPROVE (FED chemistry export) and convert to mass on filter

`volume_m3 = FlowRate_Val (L/min) × SampDur_Val (min) / 1000`, then `mass_µg = conc_µg/m³ × volume_m³`.
The xlsx is slow to parse, so the cleaned frame is cached as a pickle next to it (inside the gitignored data directory).

In [ ]:
if IMPROVE_CACHE.exists():
    imp = pd.read_pickle(IMPROVE_CACHE)
    print(f"Loaded IMPROVE cache: {len(imp):,} rows")
else:
    raw = pd.read_excel(IMPROVE_XLSX, sheet_name="Data")
    sites_meta = pd.read_excel(IMPROVE_XLSX, sheet_name="Sites")
    num_cols = ["ECf_Val", "OCf_Val", "fAbs_Val", "FlowRate_Val", "FEf_Val",
                "MF_Val", "SampDur_Val", "SOILf_Val"]
    for c in num_cols:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
        raw.loc[raw[c] <= -999, c] = np.nan   # FED missing-value sentinel
    raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
    raw = raw.drop_duplicates(subset=["Dataset", "SiteCode", "POC", "Date", "AuxID"])

    keep = [c for c in ["Code", "Site", "State", "Latitude", "Longitude"] if c in sites_meta.columns]
    sm = sites_meta[keep].drop_duplicates(subset=["Code"]).rename(
        columns={"Code": "SiteCode", "Site": "SiteName"})
    imp = raw.merge(sm, on="SiteCode", how="left")
    imp.to_pickle(IMPROVE_CACHE)
    print(f"Parsed IMPROVE export: {len(imp):,} rows (cached)")

flow_ok = imp["FlowRate_Val"].gt(0) & imp["SampDur_Val"].gt(0)
imp["volume_m3"] = np.where(flow_ok, imp["FlowRate_Val"] * imp["SampDur_Val"] / 1000.0, np.nan)
for conc, mass in [("ECf_Val", "EC_mass_ug"), ("OCf_Val", "OC_mass_ug"),
                   ("FEf_Val", "Fe_mass_ug"), ("MF_Val", "PM25_mass_ug")]:
    imp[mass] = imp[conc] * imp["volume_m3"]

n_total = len(imp)
n_fabs = imp["fAbs_Val"].gt(0).sum()
n_vol = imp["volume_m3"].notna().sum()
print(f"IMPROVE rows: {n_total:,} | sites: {imp['SiteCode'].nunique()} | "
      f"dates {imp['Date'].min().date()} -> {imp['Date'].max().date()}")
print(f"rows with positive fAbs: {n_fabs:,} | rows with computable volume: {n_vol:,} "
      f"({100 * n_vol / n_total:.1f}%)")

## 2. Load SPARTAN

Three SPARTAN views, built from the unified filter dataset:

- **`sp_pairs`** — HIPS fAbs paired with FTIR EC/OC on the *same filter* (the filters that actually enter the HIPS-vs-EC analysis). FTIR `MassLoading_ug` is a direct measure of mass on filter, no volume needed.
- **`sp_chem`** — ChemSpec thermal OC/EC per filter (method-matched to IMPROVE's thermal-optical OC/EC for the OC/EC ratio view).
- **`sp_fe`** — ChemSpec iron and PM2.5 matched to HIPS fAbs by site + sample date (the public ChemSpec rows carry no volume, so mass uses the HIPS volume of the same sampling event).

Plus **`hips_all`** — the HIPS batch file with every HIPS site (incl. ETBI), PM2.5 sample filters only.

In [ ]:
sp = pd.read_pickle(SPARTAN_PKL)
sp["SampleDate"] = pd.to_datetime(sp["SampleDate"], errors="coerce")
sp["base"] = sp["FilterId"].astype(str).str.replace(r"-\d+$", "", regex=True)


def param(name, cols=("Site", "base", "SampleDate", "Concentration", "MassLoading_ug", "Volume_m3")):
    d = sp.loc[sp["Parameter"].eq(name), list(cols)].copy()
    return d.rename(columns={"Concentration": name})


# --- (a) same-filter HIPS fAbs + FTIR EC/OC pairs ---
hips = param("HIPS_Fabs")[["Site", "base", "SampleDate", "HIPS_Fabs", "Volume_m3"]]
ec_f = param("EC_ftir").rename(columns={"MassLoading_ug": "EC_mass_ug"})
oc_f = param("OC_ftir").rename(columns={"MassLoading_ug": "OC_mass_ug"})
sp_pairs = (hips
            .merge(ec_f[["Site", "base", "SampleDate", "EC_ftir", "EC_mass_ug"]],
                   on=["Site", "base", "SampleDate"], how="inner")
            .merge(oc_f[["Site", "base", "SampleDate", "OC_ftir", "OC_mass_ug"]],
                   on=["Site", "base", "SampleDate"], how="left"))
sp_pairs = sp_pairs.dropna(subset=["HIPS_Fabs", "EC_ftir"])
sp_pairs["Fabs_EC"] = np.where(sp_pairs["EC_ftir"] > 0,
                               sp_pairs["HIPS_Fabs"] / sp_pairs["EC_ftir"], np.nan)
sp_pairs["OC_EC_ftir"] = np.where(sp_pairs["EC_ftir"] > 0,
                                  sp_pairs["OC_ftir"] / sp_pairs["EC_ftir"], np.nan)

# --- (b) ChemSpec thermal OC/EC ---
ec_t = param("ChemSpec_EC_PM2.5")[["Site", "base", "SampleDate", "ChemSpec_EC_PM2.5"]]
oc_t = param("ChemSpec_OC_PM2.5")[["Site", "base", "SampleDate", "ChemSpec_OC_PM2.5"]]
sp_chem = ec_t.merge(oc_t, on=["Site", "base", "SampleDate"], how="inner")
sp_chem["OC_EC_thermal"] = np.where(sp_chem["ChemSpec_EC_PM2.5"] > 0,
                                    sp_chem["ChemSpec_OC_PM2.5"] / sp_chem["ChemSpec_EC_PM2.5"],
                                    np.nan)

# --- (c) iron / PM2.5 matched to HIPS by site + date (same sampling event) ---
fe = param("ChemSpec_Iron_PM2.5")[["Site", "SampleDate", "ChemSpec_Iron_PM2.5"]]
pm = param("ChemSpec_Filter_PM2.5_mass")[["Site", "SampleDate", "ChemSpec_Filter_PM2.5_mass"]]
sp_fe = (hips.merge(fe, on=["Site", "SampleDate"], how="inner")
             .merge(pm, on=["Site", "SampleDate"], how="left"))
sp_fe["Fe_ugm3"] = sp_fe["ChemSpec_Iron_PM2.5"] / 1000.0          # ng/m3 -> ug/m3
sp_fe["Fe_mass_ug"] = sp_fe["Fe_ugm3"] * sp_fe["Volume_m3"]
sp_fe["PM25_mass_ug"] = sp_fe["ChemSpec_Filter_PM2.5_mass"] * sp_fe["Volume_m3"]

# --- (d) all-site HIPS batch file (incl. ETBI) ---
hips_all = pd.read_csv(HIPS_BATCH_CSV)
hips_all["SampleDate"] = pd.to_datetime(hips_all["SampleDate"], errors="coerce")
hips_all = hips_all[hips_all["FilterType"].eq("PM2.5") & hips_all["Fabs"].notna()].copy()
lookup = pd.read_excel(SITE_LOOKUP_XLSX)
site_name_map = dict(zip(lookup["SiteCode"], lookup["Description"]))

print(f"sp_pairs (same-filter fAbs+FTIR EC): {len(sp_pairs)} rows | "
      f"{sp_pairs.groupby('Site').size().to_dict()}")
print(f"sp_chem (thermal OC/EC): {len(sp_chem)} rows")
print(f"sp_fe (date-matched Fe x fAbs): {len(sp_fe)} rows | "
      f"{sp_fe.groupby('Site').size().to_dict()}")
print(f"hips_all (all HIPS sites, PM2.5 samples): {len(hips_all)} rows, "
      f"{hips_all['Site'].nunique()} sites | ETBI n={hips_all['Site'].eq('ETBI').sum()}")

## 3. Cross-plots in mass on filter (µg), red on blue

What the instrument sees is the deposit, not the airflow, so the bridge between networks is mass on filter.
All IMPROVE data stays in the plot (blue cloud) so we can see where SPARTAN (red) falls inside it.
Axes are linear — the whole point of mass space is that the ranges are finally comparable.
fAbs is left in Mm⁻¹: for the y-axis what matters is that it is the same filter-integrated optical measurement in both networks, and the per-network fits are computed in this space.

In [ ]:
def scatter_mass(ax, imp_x, imp_y, sp_df, sp_x, sp_y, xlabel, ylabel, fit=True):
    """IMPROVE cloud (blue, subsampled for drawing) + SPARTAN (red) + per-network OLS fits."""
    d_imp = pd.DataFrame({"x": imp_x, "y": imp_y}).replace([np.inf, -np.inf], np.nan).dropna()
    d_imp = d_imp[(d_imp["x"] > 0) & (d_imp["y"] > 0)]
    draw = d_imp.sample(min(len(d_imp), SCATTER_SAMPLE), random_state=0)
    ax.scatter(draw["x"], draw["y"], s=5, color=NET_COLORS["IMPROVE"], alpha=0.25,
               linewidths=0, label=f"IMPROVE (n={len(d_imp):,})", rasterized=True)

    d_sp = sp_df[[sp_x, sp_y, "Site"]].replace([np.inf, -np.inf], np.nan).dropna()
    d_sp = d_sp[(d_sp[sp_x] > 0) & (d_sp[sp_y] > 0)]
    for site, g in d_sp.groupby("Site"):
        ax.scatter(g[sp_x], g[sp_y], s=18, color=SITE_COLORS.get(site, "0.3"), alpha=0.85,
                   edgecolor="white", linewidth=0.25,
                   label=f"{site} {SITE_NAMES.get(site, '')} (n={len(g)})")

    stats_rows = []
    if fit:
        for name, dx, dy, color in [("IMPROVE", d_imp["x"], d_imp["y"], NET_COLORS["IMPROVE"]),
                                    ("SPARTAN", d_sp[sp_x], d_sp[sp_y], NET_COLORS["SPARTAN"])]:
            st = calculate_regression_stats(dx, dy)
            if st:
                xs = np.linspace(0, np.nanpercentile(dx, 99.5), 50)
                ax.plot(xs, st["slope"] * xs + st["intercept"], color=color, lw=1.6, ls="--")
                stats_rows.append({"network": name, "x": xlabel, "y": ylabel, **st})

    xmax = max(np.nanpercentile(d_imp["x"], 99.5) if len(d_imp) else 0,
               d_sp[sp_x].quantile(0.995) if len(d_sp) else 0)
    ymax = max(np.nanpercentile(d_imp["y"], 99.5) if len(d_imp) else 0,
               d_sp[sp_y].quantile(0.995) if len(d_sp) else 0)
    ax.set_xlim(0, xmax * 1.04)
    ax.set_ylim(0, ymax * 1.04)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7, loc="upper right", framealpha=0.9)
    return stats_rows


fig, axes = plt.subplots(2, 2, figsize=(13, 10.5))
all_fits = []

all_fits += scatter_mass(axes[0, 0], imp["EC_mass_ug"], imp["fAbs_Val"],
                         sp_pairs, "EC_mass_ug", "HIPS_Fabs",
                         "EC mass on filter (µg)", "fAbs (Mm⁻¹)")
axes[0, 0].set_title("fAbs vs EC mass — SPARTAN EC is FTIR, IMPROVE EC is thermal-optical")

all_fits += scatter_mass(axes[0, 1], imp["EC_mass_ug"], imp["OC_mass_ug"],
                         sp_pairs, "EC_mass_ug", "OC_mass_ug",
                         "EC mass on filter (µg)", "OC mass on filter (µg)")
axes[0, 1].set_title("OC vs EC mass (SPARTAN: FTIR masses)")

all_fits += scatter_mass(axes[1, 0], imp["Fe_mass_ug"], imp["fAbs_Val"],
                         sp_fe, "Fe_mass_ug", "HIPS_Fabs",
                         "Fe mass on filter (µg)", "fAbs (Mm⁻¹)")
axes[1, 0].set_title("fAbs vs iron mass (no EC in this plot)")

all_fits += scatter_mass(axes[1, 1], imp["PM25_mass_ug"], imp["fAbs_Val"],
                         sp_fe, "PM25_mass_ug", "HIPS_Fabs",
                         "PM₂.₅ mass on filter (µg)", "fAbs (Mm⁻¹)")
axes[1, 1].set_title("fAbs vs PM₂.₅ mass")

fig.suptitle("SPARTAN on top of IMPROVE — mass on filter, linear axes", y=1.005, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig01_mass_crossplots.png", dpi=200, bbox_inches="tight")
plt.show()

mass_fits = pd.DataFrame(all_fits)
mass_fits.to_csv(TAB_DIR / "mass_crossplot_fits.csv", index=False)
mass_fits.round(3)

### Read-out

The fits above are the cross-network bridge: in fAbs-vs-EC-mass space, the slope is an effective MAC-like
sensitivity (Mm⁻¹ per µg EC on the filter). If SPARTAN behaved like IMPROVE on the same deposit mass, the red
and blue dashed lines would lie on top of each other.

## 4. fAbs / EC ratio by site

Per-site median fAbs/EC (≈ effective MAC, m²/g — volume cancels). IMPROVE sites (≥50 valid samples) are
ranked by median; SPARTAN sites are placed in the same ranking. This is the view for "find other sites with
an Addis-like effective MAC offset".

In [ ]:
imp_valid = imp[(imp["ECf_Val"] > 0) & (imp["fAbs_Val"] > 0)].copy()
imp_valid["Fabs_EC"] = imp_valid["fAbs_Val"] / imp_valid["ECf_Val"]
imp_valid["OC_EC"] = np.where(imp_valid["OCf_Val"] > 0,
                              imp_valid["OCf_Val"] / imp_valid["ECf_Val"], np.nan)


def site_ratio_table(df, site_col, ratio_col, min_n=MIN_SITE_N):
    g = df[[site_col, ratio_col]].dropna().groupby(site_col)[ratio_col]
    t = g.agg(n="size", median="median",
              p25=lambda s: s.quantile(0.25), p75=lambda s: s.quantile(0.75))
    return t[t["n"] >= min_n].sort_values("median").reset_index()


imp_fabs_ec = site_ratio_table(imp_valid, "SiteCode", "Fabs_EC")
sp_fabs_ec = site_ratio_table(sp_pairs, "Site", "Fabs_EC", min_n=10)

fig, ax = plt.subplots(figsize=(13, 5.5))
x = np.arange(len(imp_fabs_ec))
ax.errorbar(x, imp_fabs_ec["median"],
            yerr=[imp_fabs_ec["median"] - imp_fabs_ec["p25"],
                  imp_fabs_ec["p75"] - imp_fabs_ec["median"]],
            fmt="o", ms=3, color="0.55", ecolor="0.8", elinewidth=0.7,
            label=f"IMPROVE sites (n={len(imp_fabs_ec)}, median ± IQR)")
for _, r in sp_fabs_ec.iterrows():
    rank = np.searchsorted(imp_fabs_ec["median"].values, r["median"])
    ax.errorbar([rank], [r["median"]],
                yerr=[[r["median"] - r["p25"]], [r["p75"] - r["median"]]],
                fmt="*", ms=17, color=SITE_COLORS[r["Site"]], ecolor=SITE_COLORS[r["Site"]],
                elinewidth=1.3, zorder=5,
                label=f"{r['Site']} {SITE_NAMES[r['Site']]} ({r['median']:.1f} m²/g)")
ax.axhline(10, color="0.25", ls=":", lw=1.1, label="MAC = 10 m²/g (calibration reference)")
ax.set_xlabel("IMPROVE sites, ranked by median fAbs/EC")
ax.set_ylabel("fAbs / EC (m²/g ≈ effective MAC)")
ax.set_title("fAbs/EC by site — SPARTAN stars placed at their rank within the IMPROVE ordering\n"
             "(SPARTAN EC = FTIR on the same filter; IMPROVE EC = thermal-optical)")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig02_fabs_ec_ratio_by_site.png", dpi=200, bbox_inches="tight")
plt.show()

sp_fabs_ec

## 5. OC / EC ratio by site

Same per-site ranking for OC/EC. SPARTAN is shown twice on purpose:
- **thermal OC/EC** (ChemSpec) — method-matched to IMPROVE's thermal-optical carbon, and
- **FTIR OC/EC** (same filters as the HIPS pairing).

Satoshi's point: FTIR infers EC partly through functional groups, calibrated on US ambient samples.
A site with much lower OC/EC than the calibration population (Addis) may not transfer well — that is the
alternative-to-char hypothesis for why FTIR EC might be off.

In [ ]:
imp_oc_ec = site_ratio_table(imp_valid, "SiteCode", "OC_EC")
sp_oc_ec_th = site_ratio_table(sp_chem, "Site", "OC_EC_thermal", min_n=10)
sp_oc_ec_ftir = site_ratio_table(sp_pairs, "Site", "OC_EC_ftir", min_n=10)

fig, ax = plt.subplots(figsize=(13, 5.5))
x = np.arange(len(imp_oc_ec))
ax.errorbar(x, imp_oc_ec["median"],
            yerr=[imp_oc_ec["median"] - imp_oc_ec["p25"], imp_oc_ec["p75"] - imp_oc_ec["median"]],
            fmt="o", ms=3, color="0.55", ecolor="0.8", elinewidth=0.7,
            label=f"IMPROVE sites (n={len(imp_oc_ec)}, median ± IQR)")
for tab, marker, suffix in [(sp_oc_ec_th, "*", "thermal"), (sp_oc_ec_ftir, "D", "FTIR")]:
    for _, r in tab.iterrows():
        rank = np.searchsorted(imp_oc_ec["median"].values, r["median"])
        ax.scatter([rank], [r["median"]], marker=marker, s=240 if marker == "*" else 70,
                   color=SITE_COLORS[r["Site"]], zorder=5, edgecolor="white", linewidth=0.5,
                   label=f"{r['Site']} {SITE_NAMES[r['Site']]} {suffix} ({r['median']:.2f})")
ax.set_xlabel("IMPROVE sites, ranked by median OC/EC")
ax.set_ylabel("OC / EC")
ax.set_title("OC/EC by site — every SPARTAN site sits below almost the entire IMPROVE network\n"
             "(stars = SPARTAN thermal OC/EC, diamonds = SPARTAN FTIR OC/EC)")
ax.legend(fontsize=7.5, ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig03_oc_ec_ratio_by_site.png", dpi=200, bbox_inches="tight")
plt.show()

pd.concat([sp_oc_ec_th.assign(method="thermal"), sp_oc_ec_ftir.assign(method="FTIR")])

## 6. The Addis-analog map: site-median OC/EC vs fAbs/EC

Both ratios in one plane. Addis (ETAD) is characterized by **low OC/EC** and an effective MAC that does not
behave like the calibration (slope ≈ 4 with a large offset instead of ≈ 10). IMPROVE sites that fall closest
to ETAD in this plane are the candidates for "another example" of the effect — sites worth pulling FTIR
spectra for. Distance is computed in log space (both ratios are scale quantities), standardized by the
IMPROVE spread.

In [ ]:
site_plane = imp_fabs_ec.rename(columns={"median": "Fabs_EC_med"})[["SiteCode", "n", "Fabs_EC_med"]].merge(
    imp_oc_ec.rename(columns={"median": "OC_EC_med"})[["SiteCode", "OC_EC_med"]],
    on="SiteCode")
sm = imp[["SiteCode", "SiteName", "State"]].drop_duplicates("SiteCode") if "SiteName" in imp.columns else None
if sm is not None:
    site_plane = site_plane.merge(sm, on="SiteCode", how="left")

sp_plane = sp_fabs_ec.rename(columns={"median": "Fabs_EC_med"})[["Site", "Fabs_EC_med"]].merge(
    sp_oc_ec_th.rename(columns={"median": "OC_EC_med"})[["Site", "OC_EC_med"]], on="Site")

etad = sp_plane[sp_plane["Site"].eq("ETAD")].iloc[0]
lx = np.log(site_plane["OC_EC_med"]); ly = np.log(site_plane["Fabs_EC_med"])
dist = np.hypot((lx - np.log(etad["OC_EC_med"])) / lx.std(),
                (ly - np.log(etad["Fabs_EC_med"])) / ly.std())
site_plane["dist_to_ETAD"] = dist
candidates = site_plane.sort_values("dist_to_ETAD").head(15)
candidates.to_csv(TAB_DIR / "addis_analog_candidates.csv", index=False)

fig, ax = plt.subplots(figsize=(9.5, 7))
ax.scatter(site_plane["OC_EC_med"], site_plane["Fabs_EC_med"], s=18, color="0.6", alpha=0.7,
           label=f"IMPROVE sites (n={len(site_plane)})")
ax.scatter(candidates["OC_EC_med"], candidates["Fabs_EC_med"], s=46, facecolor="none",
           edgecolor="#7B241C", linewidth=1.2, label="15 nearest to ETAD")
for _, r in candidates.head(8).iterrows():
    ax.annotate(r["SiteCode"], (r["OC_EC_med"], r["Fabs_EC_med"]),
                xytext=(4, 3), textcoords="offset points", fontsize=7, color="#7B241C")
for _, r in sp_plane.iterrows():
    ax.scatter(r["OC_EC_med"], r["Fabs_EC_med"], marker="*", s=380,
               color=SITE_COLORS[r["Site"]], edgecolor="black", linewidth=0.6, zorder=5,
               label=f"{r['Site']} {SITE_NAMES[r['Site']]}")
ax.axvline(etad["OC_EC_med"], color=SITE_COLORS["ETAD"], lw=0.9, ls=":")
ax.axhline(etad["Fabs_EC_med"], color=SITE_COLORS["ETAD"], lw=0.9, ls=":")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("site median OC / EC")
ax.set_ylabel("site median fAbs / EC (m²/g)")
ax.set_title("Site-median ratio plane — IMPROVE sites closest to Addis are FTIR-spectra candidates")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig04_addis_analog_map.png", dpi=200, bbox_inches="tight")
plt.show()

candidates

## 7. HIPS fAbs vs iron — no EC anywhere in the plot

Iron absorbs in the red and is known to matter for HIPS, and these are dusty/industrial sites — yet the
site-level cross-plots have never shown an iron signal at the SPARTAN sites. The mass-space version is the
check that conclusion survives the new framing (this duplicates panel C of fig01 but adds per-site fits and
the Fe fraction context). Not the current focus — just confirming nothing was hiding behind the volume scaling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

ax = axes[0]
d_imp = imp[(imp["Fe_mass_ug"] > 0) & (imp["fAbs_Val"] > 0)]
draw = d_imp.sample(min(len(d_imp), SCATTER_SAMPLE), random_state=0)
ax.scatter(draw["Fe_mass_ug"], draw["fAbs_Val"], s=5, color=NET_COLORS["IMPROVE"], alpha=0.25,
           linewidths=0, label=f"IMPROVE (n={len(d_imp):,})", rasterized=True)
fe_fit_rows = []
st = calculate_regression_stats(d_imp["Fe_mass_ug"], d_imp["fAbs_Val"])
if st:
    fe_fit_rows.append({"group": "IMPROVE", **st})
for site, g in sp_fe.dropna(subset=["Fe_mass_ug", "HIPS_Fabs"]).groupby("Site"):
    ax.scatter(g["Fe_mass_ug"], g["HIPS_Fabs"], s=20, color=SITE_COLORS.get(site, "0.3"),
               alpha=0.85, edgecolor="white", linewidth=0.25,
               label=f"{site} {SITE_NAMES.get(site, '')} (n={len(g)})")
    st = calculate_regression_stats(g["Fe_mass_ug"], g["HIPS_Fabs"])
    if st:
        fe_fit_rows.append({"group": site, **st})
ax.set_xlim(0, max(d_imp["Fe_mass_ug"].quantile(0.995), sp_fe["Fe_mass_ug"].quantile(0.995)) * 1.05)
ax.set_ylim(0, max(d_imp["fAbs_Val"].quantile(0.995), sp_fe["HIPS_Fabs"].quantile(0.995)) * 1.05)
ax.set_xlabel("Fe mass on filter (µg)")
ax.set_ylabel("fAbs (Mm⁻¹)")
ax.set_title("HIPS fAbs vs iron mass")
ax.legend(fontsize=7.5)

ax = axes[1]
for site, g in sp_fe.iterrows() if False else sp_fe.groupby("Site"):
    frac = (100 * g["Fe_ugm3"] / g["ChemSpec_Filter_PM2.5_mass"]).replace([np.inf, -np.inf], np.nan).dropna()
    ax.scatter(frac, g.loc[frac.index, "HIPS_Fabs"], s=20, color=SITE_COLORS.get(site, "0.3"),
               alpha=0.85, edgecolor="white", linewidth=0.25,
               label=f"{site} {SITE_NAMES.get(site, '')}")
ax.set_xlabel("Fe fraction of PM₂.₅ (%)")
ax.set_ylabel("fAbs (Mm⁻¹)")
ax.set_title("SPARTAN: fAbs vs iron fraction of mass")
ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "fig05_hips_vs_iron.png", dpi=200, bbox_inches="tight")
plt.show()

fe_fits = pd.DataFrame(fe_fit_rows)
fe_fits.to_csv(TAB_DIR / "fabs_vs_iron_fits.csv", index=False)
fe_fits.round(3)

## 8. The new ETBI (Bishoftu, Ethiopia) site

ETBI has only HIPS so far (~2 dozen PM2.5 filters since Oct 2025; no public ChemSpec or FTIR yet), so
ratio plots are not possible. What we *can* check: does the second Ethiopian site land in the same fAbs
regime as Addis? If ETBI shows the same high-fAbs behavior, that supports a real regional aerosol regime
rather than an ETAD-specific instrument/handling story — and it makes ETBI the natural confirmation site
once its carbon data arrive.

In [ ]:
site_order = (hips_all.groupby("Site")["Fabs"].median().sort_values().index.tolist())

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.6), gridspec_kw={"width_ratios": [1.7, 1]})
ax = axes[0]
data = [hips_all.loc[hips_all["Site"].eq(s), "Fabs"].dropna().values for s in site_order]
bp = ax.boxplot(data, labels=site_order, showfliers=False, patch_artist=True)
for patch, s in zip(bp["boxes"], site_order):
    if s in ("ETAD", "ETBI"):
        patch.set_facecolor(SITE_COLORS[s]); patch.set_alpha(0.95)
    elif s in SITE_COLORS:
        patch.set_facecolor(SITE_COLORS[s]); patch.set_alpha(0.55)
    else:
        patch.set_facecolor("0.85")
counts = hips_all.groupby("Site").size()
ax.set_xticklabels([f"{s}\n(n={counts[s]})" for s in site_order], fontsize=7)
ax.set_ylabel("HIPS fAbs (Mm⁻¹)")
ax.set_title("HIPS fAbs across all SPARTAN sites (PM₂.₅ sample filters) — Ethiopia highlighted")

ax = axes[1]
for s in ("ETAD", "ETBI"):
    v = np.sort(hips_all.loc[hips_all["Site"].eq(s), "Fabs"].dropna().values)
    ax.plot(v, np.linspace(0, 1, len(v)), color=SITE_COLORS[s], lw=2,
            label=f"{s} {SITE_NAMES[s]} (n={len(v)}, median {np.median(v):.1f})")
ax.set_xlabel("HIPS fAbs (Mm⁻¹)")
ax.set_ylabel("empirical CDF")
ax.set_title("ETBI vs ETAD fAbs distribution")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig06_etbi_fabs_context.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
etbi = hips_all[hips_all["Site"].eq("ETBI")].sort_values("SampleDate")
etad_recent = hips_all[hips_all["Site"].eq("ETAD")].sort_values("SampleDate")

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
ax = axes[0]
ax.plot(etbi["SampleDate"], etbi["Fabs"], "o-", color=SITE_COLORS["ETBI"], ms=5, lw=0.9,
        label="ETBI Bishoftu")
ax.axhline(etad_recent["Fabs"].median(), color=SITE_COLORS["ETAD"], ls="--", lw=1.3,
           label=f"ETAD median ({etad_recent['Fabs'].median():.1f})")
ax.set_ylabel("HIPS fAbs (Mm⁻¹)")
ax.set_title("ETBI fAbs time series (commissioning, Oct–Dec 2025)")
ax.legend(fontsize=8)
fig.autofmt_xdate()

ax = axes[1]
for s, df_s in [("ETAD", etad_recent), ("ETBI", etbi)]:
    d = df_s.dropna(subset=["tau", "Fabs"])
    ax.scatter(d["tau"], d["Fabs"], s=22, color=SITE_COLORS[s], alpha=0.8,
               edgecolor="white", linewidth=0.3, label=f"{s} (n={len(d)})")
ax.set_xlabel("HIPS tau (optical depth)")
ax.set_ylabel("fAbs (Mm⁻¹)")
ax.set_title("fAbs vs tau — ETBI lies on the same optical curve as ETAD")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig07_etbi_timeseries_tau.png", dpi=200, bbox_inches="tight")
plt.show()

etbi_summary = (hips_all.groupby("Site")["Fabs"]
                .agg(n="size", median="median", p25=lambda s: s.quantile(0.25),
                     p75=lambda s: s.quantile(0.75), max="max")
                .round(2).sort_values("median", ascending=False))
etbi_summary.insert(0, "site_name", [site_name_map.get(s, "") for s in etbi_summary.index])
etbi_summary.to_csv(TAB_DIR / "hips_fabs_all_sites_summary.csv")
etbi_summary.head(12)

## 9. Saved artifacts and per-site ratio table

In [ ]:
imp_fabs_ec.assign(network="IMPROVE", ratio="Fabs_EC").rename(columns={"SiteCode": "site"}).pipe(
    lambda a: pd.concat([
        a,
        imp_oc_ec.assign(network="IMPROVE", ratio="OC_EC").rename(columns={"SiteCode": "site"}),
        sp_fabs_ec.assign(network="SPARTAN", ratio="Fabs_EC").rename(columns={"Site": "site"}),
        sp_oc_ec_th.assign(network="SPARTAN", ratio="OC_EC_thermal").rename(columns={"Site": "site"}),
        sp_oc_ec_ftir.assign(network="SPARTAN", ratio="OC_EC_ftir").rename(columns={"Site": "site"}),
    ], ignore_index=True)
).to_csv(TAB_DIR / "site_ratio_stats.csv", index=False)

print("Artifacts written:")
for p in sorted(FIG_DIR.glob("*.png")) + sorted(TAB_DIR.glob("*.csv")):
    print(" -", p.relative_to(REPO_ROOT))